#Configuration



In [ ]:
import os
import pandas as pd
import requests
import json
import base64
import time
import glob

# DataForSEO Credentials
login = os.environ["DATAFORSEO_LOGIN"]
password = os.environ["DATAFORSEO_PASSWORD"]

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

base_path = "/content/drive/MyDrive/RA/health_care/dentist_LMS_keywords"
if not os.path.exists(base_path):
    os.makedirs(base_path)

# Chunk data by category


In [ ]:
import pandas as pd
import numpy as np
import os

base_path = '/content/drive/MyDrive/RA/health_care/dentist_LMS_keywords/output/final/'
main_file = base_path + 'final_merged_time.csv'
reviews_file = base_path + 'all_reviews_detailed.csv'

cat_mapping = {
    'keywords_General_Dentist': 'General',
    'keywords_Special_Dentist': 'Special',
    'keywords_Surgery_Dentist': 'Surgery'
}

df_main = pd.read_csv(main_file)


def clean_zip_code(val):
    val = str(val).strip()
    if '-' in val: val = val.split('-')[0]
    val = ''.join(filter(str.isdigit, val))
    val = val[:5]
    if val: return float(val)
    return np.nan


if 'zip' in df_main.columns:
    temp_main_zip = df_main['zip'].apply(clean_zip_code)
else:
    temp_main_zip = pd.Series([np.nan] * len(df_main))

df_main['temp_match_zip'] = temp_main_zip.astype(str).str.replace(r'\.0$', '', regex=True).str.strip()

before_drop = len(df_main)
df_main = df_main.drop_duplicates(subset=['title', 'temp_match_zip'], keep='first')
print(f"Deduplicate: delete {before_drop - len(df_main)}，remain {len(df_main)}")

main_zip_str = df_main['temp_match_zip'].copy()

df_main = df_main.drop(columns=['temp_match_zip'])


mapping_df = pd.DataFrame({
    'match_title': df_main['title'],
    'match_zip': main_zip_str,
    'match_category': df_main['category']
}).drop_duplicates()


for original_cat, file_suffix in cat_mapping.items():
    main_subset = df_main[df_main['category'] == original_cat]
    main_save_path = f"{base_path}final_merged_time_{file_suffix}.csv"
    main_subset.to_csv(main_save_path, index=False)
    print(f"Saved: final_merged_time_{file_suffix}.csv ({len(main_subset)}lines)")



chunk_size = 50000
first_chunk_flag = {cat: True for cat in cat_mapping.values()}
counts = {cat: 0 for cat in cat_mapping.values()}
chunk_count = 0

for chunk in pd.read_csv(reviews_file, chunksize=chunk_size, low_memory=False):
    chunk_count += 1
    if chunk_count % 10 == 0:
        print(f" Process {chunk_count * chunk_size}.")

    reviews_zip_str = chunk['shop_zip'].astype(str).str.replace(r'\.0$', '', regex=True).str.strip()

    chunk_temp = chunk.copy()
    chunk_temp['match_shop_zip'] = reviews_zip_str

    merged_chunk = pd.merge(
        chunk_temp,
        mapping_df,
        left_on=['shop_title', 'match_shop_zip'],
        right_on=['match_title', 'match_zip'],
        how='inner'
    )

    for original_cat, file_suffix in cat_mapping.items():
        cat_subset = merged_chunk[merged_chunk['match_category'] == original_cat].copy()

        if not cat_subset.empty:
            cols_to_drop = ['match_shop_zip', 'match_title', 'match_zip', 'match_category']
            cat_subset = cat_subset.drop(columns=cols_to_drop, errors='ignore')

            review_save_path = f"{base_path}all_reviews_detailed_{file_suffix}.csv"

            mode = 'w' if first_chunk_flag[file_suffix] else 'a'
            header = first_chunk_flag[file_suffix]
            cat_subset.to_csv(review_save_path, mode=mode, header=header, index=False)

            first_chunk_flag[file_suffix] = False
            counts[file_suffix] += len(cat_subset)

for file_suffix, count in counts.items():
    print(f"Saved: all_reviews_detailed_{file_suffix}.csv (lines: {count})")


print("\n" + "="*50)

for file_suffix in cat_mapping.values():
    main_output_path = f"{base_path}final_merged_time_{file_suffix}.csv"
    print(f"\n{main_output_path.split('/')[-1]}")
    display_main = pd.read_csv(main_output_path, nrows=5)
    print(display_main.head())

    review_output_path = f"{base_path}all_reviews_detailed_{file_suffix}.csv"
    print(f"\n{review_output_path.split('/')[-1]}")
    try:
        display_review = pd.read_csv(review_output_path, nrows=5)
        print(display_review.head())
    except pd.errors.EmptyDataError:
        print("empty")

    print("-" * 50)

# Analysis

## Raw Data

In [ ]:
import pandas as pd
import numpy as np
from math import radians, sin, cos, sqrt, asin

df = pd.read_csv('/content/drive/MyDrive/RA/health_care/dentist_LMS_keywords/output/final/final_merged_time.csv')

if not df.empty:
    def clean_zip_code(val):
        val = str(val).strip()
        if '-' in val: val = val.split('-')[0]
        val = ''.join(filter(str.isdigit, val))
        val = val[:5]
        if val: return float(val)
        return np.nan

    if 'zip' in df.columns:
        df['zip_clean'] = df['zip'].apply(clean_zip_code)
    else:
        df['zip_clean'] = np.nan

    def map_zip_to_location(z):
        if pd.isna(z): return 'Unknown'
        try: z = int(z)
        except ValueError: return 'Unknown'
        if z == 12953: return 'Malone_NY_S'
        if z == 12983: return 'SaranacLake_NY_S'
        if 13200 <= z <= 13299: return 'Syracuse_NY_M'
        if 14200 <= z <= 14299: return 'Buffalo_NY_L'
        if (10000 <= z <= 10499) or (11000 <= z <= 11699) or (10300 <= z <= 10399): return 'NYC_NY_L'
        if z in [95501, 95502, 95503, 95534]: return 'Eureka_CA_S'
        if z == 95437: return 'FortBragg_CA_S'
        if 95350 <= z <= 95358: return 'Modesto_CA_M'
        if 94100 <= z <= 94188: return 'SanFrancisco_CA_L'
        if (90000 <= z <= 91699): return 'LA_CA_L'
        if z in [30474, 30475]: return 'Vidalia_GA_S'
        if z == 30577: return 'Toccoa_GA_S'
        if 31200 <= z <= 31299: return 'Macon_GA_M'
        if 30900 <= z <= 30999: return 'Augusta_GA_L'
        if (30300 <= z <= 30399) or (31100 <= z <= 31199): return 'Atlanta_GA_L'
        return 'Unknown'

    df['mapped_location'] = df['zip_clean'].apply(map_zip_to_location)
    df = df[df['mapped_location'] != 'Unknown'].copy()

    cat_mapping = {
        'keywords_General_Dentist': 'General',
        'keywords_Special_Dentist': 'Special',
        'keywords_Surgery_Dentist': 'Surgery'
    }

    if 'category' in df.columns:
        df = df[df['category'].isin(cat_mapping.keys())].copy()
        df['Keyword_Category'] = df['category'].map(cat_mapping)
    else:
        df['Keyword_Category'] = 'Unknown'

    def get_loc_type(loc):
        if loc.endswith('_S'): return 'Small'
        if loc.endswith('_M'): return 'Mid_Size'
        if loc.endswith('_L'): return 'Large'
        return 'Unknown'
    df['Location_Type'] = df['mapped_location'].apply(get_loc_type)

    df['Location_Type'] = pd.Categorical(
        df['Location_Type'],
        categories=['Mid_Size', 'Large', 'Small'],
        ordered=False
    )

    df['Category_Location'] = df['Keyword_Category'].astype(str) + '_' + df['Location_Type'].astype(str)

    df['Category_Location'] = pd.Categorical(
        df['Category_Location'],
        categories=[
            'General_Mid_Size',
            'General_Large', 'General_Small',
            'Special_Mid_Size', 'Special_Large', 'Special_Small',
            'Surgery_Mid_Size', 'Surgery_Large', 'Surgery_Small'
        ],
        ordered=False
    )

    hub_coords = {
        'NYC_NY_L': (40.7128, -74.0060), 'Buffalo_NY_L': (42.8864, -78.8784), 'Syracuse_NY_M': (43.0481, -76.1474),
        'SaranacLake_NY_S': (44.3295, -74.1313), 'Malone_NY_S': (44.8487, -74.2963),
        'LA_CA_L': (34.0522, -118.2437), 'SanFrancisco_CA_L': (37.7749, -122.4194), 'Modesto_CA_M': (37.6391, -120.9969),
        'Eureka_CA_S': (40.8021, -124.1637), 'FortBragg_CA_S': (39.4457, -123.8053),
        'Atlanta_GA_L': (33.7490, -84.3880), 'Augusta_GA_L': (33.4735, -81.9665), 'Macon_GA_M': (32.8407, -83.6324),
        'Vidalia_GA_S': (32.2177, -82.4135), 'Toccoa_GA_S': (34.5771, -83.3324)
    }

    def get_distance(row):
        loc = row['mapped_location']
        if loc not in hub_coords: return np.nan
        h_lat, h_lon = hub_coords[loc]
        try:
            lon1, lat1, lon2, lat2 = map(radians, [float(row['longitude']), float(row['latitude']), h_lon, h_lat])
            dlon = lon2 - lon1
            dlat = lat2 - lat1
            a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
            c = 2 * asin(sqrt(a))
            return c * 3958.8
        except: return np.nan

    if 'latitude' in df.columns:
        df['distance_to_hub'] = df.apply(get_distance, axis=1)

    df['zip_clinic_count'] = df.groupby('zip_clean')['zip_clean'].transform('count')

    def calc_peer_dist(sub):
        if len(sub) < 2: return np.zeros(len(sub))
        lats = np.radians(sub['latitude'].values)
        lons = np.radians(sub['longitude'].values)
        dlat = lats[:, None] - lats[None, :]
        dlon = lons[:, None] - lons[None, :]
        a = np.sin(dlat/2)**2 + np.cos(lats[:, None]) * np.cos(lats[None, :]) * np.sin(dlon/2)**2
        c = 2 * np.arcsin(np.sqrt(a))
        dists = c * 3958.8
        return np.sum(dists, axis=1) / (len(sub) - 1)

    coords = df[['latitude', 'longitude', 'zip_clean']].dropna()
    peer_dist_map = {}
    for z, group in coords.groupby('zip_clean'):
        if len(group) > 1:
            dists = calc_peer_dist(group)
            for idx, val in zip(group.index, dists):
                peer_dist_map[idx] = val
        else:
            for idx in group.index:
                peer_dist_map[idx] = 0.0

    df['avg_peer_dist'] = df.index.map(peer_dist_map).fillna(0)
    df['avg_peer_dist'] = df['avg_peer_dist'].replace(0.0, np.nan)

    df['rating_value'] = pd.to_numeric(df['rating_value'], errors='coerce')

    df_low = df[df['rating_value'] <= 3.0].copy()
    peer_dist_low_map = {}
    if not df_low.empty:
        coords_low = df_low[['latitude', 'longitude', 'zip_clean']].dropna()
        for z, group in coords_low.groupby('zip_clean'):
            if len(group) > 1:
                dists = calc_peer_dist(group)
                for idx, val in zip(group.index, dists):
                    peer_dist_low_map[idx] = val
            else:
                for idx in group.index:
                    peer_dist_low_map[idx] = 0.0

    df['avg_dist_to_1star_peers'] = df.index.map(peer_dist_low_map)
    df['avg_dist_to_1star_peers'] = df['avg_dist_to_1star_peers'].replace(0.0, np.nan)

    df_other = df[df['rating_value'] > 3.0].copy()
    peer_dist_other_map = {}
    if not df_other.empty:
        coords_other = df_other[['latitude', 'longitude', 'zip_clean']].dropna()
        for z, group in coords_other.groupby('zip_clean'):
            if len(group) > 1:
                dists = calc_peer_dist(group)
                for idx, val in zip(group.index, dists):
                    peer_dist_other_map[idx] = val
            else:
                for idx in group.index:
                    peer_dist_other_map[idx] = 0.0

    df['avg_dist_to_other_peers'] = df.index.map(peer_dist_other_map)
    df['avg_dist_to_other_peers'] = df['avg_dist_to_other_peers'].replace(0.0, np.nan)

    df['log_votes'] = np.log1p(pd.to_numeric(df['votes_count'], errors='coerce').fillna(0))
    df['is_low_rating'] = np.where(df['rating_value'] <= 3, 1, 0)

    def get_state(loc):
        if '_NY_' in loc: return 'NY'
        if '_CA_' in loc: return 'CA'
        if '_GA_' in loc: return 'GA'
        return 'Other'
    df['State'] = df['mapped_location'].apply(get_state)

    reg_df = df.dropna(subset=[
        'rating_value', 'distance_to_hub', 'log_votes',
        'zip_clinic_count', 'State', 'Category_Location', 'Location_Type'
    ]).copy()

    print(f"N = {len(reg_df)}")

In [ ]:
import pandas as pd

df_rated = df.dropna(subset=['rating_value']).copy()

low_group = df_rated[df_rated['rating_value'] <= 3.0]
high_group = df_rated[df_rated['rating_value'] > 3.0]

low_total = len(low_group)
low_with_time = low_group['start_date'].notna().sum()
low_prop = (low_with_time / low_total) if low_total > 0 else 0

high_total = len(high_group)
high_with_time = high_group['start_date'].notna().sum()
high_prop = (high_with_time / high_total) if high_total > 0 else 0

print(f"N= {len(df_rated)}")
print("-" * 40)
print(f"N_low= {low_total}")
print(f"N_low with start_date= {low_with_time}")
print(f"Coverage: {low_prop:.2%}")
print("-" * 40)
print(f"N_high= {high_total}")
print(f"N_high with start_date= {high_with_time}")
print(f"Coverage: {high_prop:.2%}")

low_missing_date_df = low_group[low_group['start_date'].isna()]
low_missing_date_clinics = low_missing_date_df['title'].tolist()

print("-" * 40)
print(f"Clinics in low group missing start_date (N={len(low_missing_date_clinics)}):")
for clinic in low_missing_date_clinics:
    print(clinic)

评分少且一般是连锁或几家店叫一个名字，网站无法精准对应

In [ ]:
low_group = df[df['rating_value'] <= 3.0]
high_group = df[df['rating_value'] > 3.0]

mean_hub_low = low_group['distance_to_hub'].mean()
mean_hub_high = high_group['distance_to_hub'].mean()
mean_hub_all = df['distance_to_hub'].mean()

mean_peer_all_low = low_group['avg_peer_dist'].mean()
mean_peer_all_high = high_group['avg_peer_dist'].mean()
mean_peer_all = df['avg_peer_dist'].mean()

mean_peer_low_low = low_group['avg_dist_to_1star_peers'].mean()
mean_peer_high_high = high_group['avg_dist_to_other_peers'].mean()

print(f"Avg Dist to Hub:")
print(f"   - Low Rate Clinics:  {mean_hub_low:.2f} miles")
print(f"   - High Rate Clinics: {mean_hub_high:.2f} miles")
print(f"   - All Clinics:       {mean_hub_all:.2f} miles")

print(f"\nAvg Dist to Peers (All peers in zip):")
print(f"   - Low Rate Clinics:  {mean_peer_all_low:.2f} miles")
print(f"   - High Rate Clinics: {mean_peer_all_high:.2f} miles")
print(f"   - All Clinics:       {mean_peer_all:.2f} miles")

print(f"\nAvg Dist to Same-Tier Peers:")
print(f"   - Low Rate to Low Rate:   {mean_peer_low_low:.2f} miles")
print(f"   - High Rate to High Rate: {mean_peer_high_high:.2f} miles")

In [ ]:
stats = []
for loc, group in df.groupby('mapped_location'):
    total_clinics = len(group)
    low_clinics = len(group[group['rating_value'] <= 3.0])
    low_pct = low_clinics / total_clinics if total_clinics > 0 else 0
    avg_low_to_low = group[group['rating_value'] <= 3.0]['avg_dist_to_1star_peers'].mean()

    stats.append({
        'Region': loc,
        'Total_Clinics': total_clinics,
        'Low_Rate_Clinics': low_clinics,
        'Low_Rate_Pct': low_pct,
        'Avg_Dist_Low_to_Low': avg_low_to_low
    })

res_df = pd.DataFrame(stats).sort_values('Low_Rate_Clinics', ascending=False)
print(res_df.to_string(index=False))

In [ ]:
import pandas as pd
import numpy as np
from sklearn.cluster import DBSCAN
from sklearn.neighbors import BallTree

df['rating_value'] = pd.to_numeric(df['rating_value'], errors='coerce')
df_valid = df.dropna(subset=['latitude', 'longitude', 'rating_value']).copy()

df_low = df_valid[df_valid['rating_value'] <= 3.0].copy()

coords_low = np.radians(df_low[['latitude', 'longitude']].values)
coords_all = np.radians(df_valid[['latitude', 'longitude']].values)

db = DBSCAN(eps=2.0/3958.8, min_samples=2, metric='haversine').fit(coords_low)
df_low['cluster'] = db.labels_
clusters = df_low[df_low['cluster'] != -1]

results = []
tree_all = BallTree(coords_all, metric='haversine')

for c_id, group in clusters.groupby('cluster'):
    center_lat = group['latitude'].mean()
    center_lon = group['longitude'].mean()
    n_low = len(group)

    if n_low > 1:
        lats = np.radians(group['latitude'].values)
        lons = np.radians(group['longitude'].values)
        dlat = lats[:, None] - lats[None, :]
        dlon = lons[:, None] - lons[None, :]
        a = np.sin(dlat/2)**2 + np.cos(lats[:, None]) * np.cos(lats[None, :]) * np.sin(dlon/2)**2
        dists = 2 * np.arcsin(np.sqrt(a)) * 3958.8
        avg_dist = np.sum(dists) / (n_low * (n_low - 1))
    else:
        avg_dist = 0

    centroid_rad = np.radians([[center_lat, center_lon]])

    n_total_2mi = tree_all.query_radius(centroid_rad, r=2.0/3958.8, count_only=True)[0]
    n_total_5mi = tree_all.query_radius(centroid_rad, r=5.0/3958.8, count_only=True)[0]

    results.append({
        'Cluster_ID': c_id,
        'Center_Lat': round(center_lat, 4),
        'Center_Lon': round(center_lon, 4),
        'Low_Clinics_Count': n_low,
        'Avg_Dist_Low_to_Low': round(avg_dist, 4),
        'Total_Clinics_2mi': n_total_2mi,
        'Low_Pct_2mi': round(n_low / n_total_2mi, 4) if n_total_2mi > 0 else 0,
        'Total_Clinics_5mi': n_total_5mi,
        'Low_Pct_5mi': round(n_low / n_total_5mi, 4) if n_total_5mi > 0 else 0
    })

res_df = pd.DataFrame(results).sort_values(by=['Low_Pct_2mi', 'Low_Clinics_Count'], ascending=[False, False])
print(res_df.to_string(index=False))

In [ ]:
import pandas as pd

data = [
    ["Macon, GA (West / Eisenhower Pkwy)", "32.8205, -83.6922", "4", "13", "30.77%", "76", "5.26%"],
    ["Atlanta, GA (South Fulton)", "33.7240, -84.5083", "2", "7", "28.57%", "31", "6.45%"],
    ["Augusta, GA (South / Richmond County)", "33.4217, -82.0223", "3", "13", "23.08%", "76", "3.95%"],
    ["Buffalo, NY (Downtown)", "42.9002, -78.8673", "11", "48", "22.92%", "110", "10.00%"],
    ["Amherst, NY (UB North Campus)", "42.9624, -78.8001", "10", "49", "20.41%", "167", "5.99%"],
    ["Buffalo, NY (North Buffalo / Parkside)", "42.9251, -78.8318", "3", "19", "15.79%", "178", "1.69%"],
    ["Eureka, CA (Old Town / Downtown)", "40.7919, -124.1462", "10", "65", "15.38%", "68", "14.71%"],
    ["Atlanta, GA (Downtown)", "33.7642, -84.3880", "15", "99", "15.15%", "169", "8.88%"],
    ["Augusta, GA (Medical District)", "33.4705, -81.9884", "5", "38", "13.16%", "68", "7.35%"],
    ["Augusta, GA (Summerville)", "33.4603, -82.0399", "2", "17", "11.76%", "140", "1.43%"],
    ["Macon, GA (Downtown Historic District)", "32.8373, -83.6327", "5", "43", "11.63%", "74", "6.76%"],
    ["Buffalo, NY (South Buffalo)", "42.8506, -78.8101", "2", "19", "10.53%", "110", "1.82%"],
    ["Smyrna, GA (Cumberland)", "33.8816, -84.4598", "2", "19", "10.53%", "75", "2.67%"],
    ["Modesto, CA (Central / Main Rd)", "37.6676, -121.0051", "11", "120", "9.17%", "180", "6.11%"],
    ["San Francisco, CA (Hayes Valley / Alamo Square)", "37.7766, -122.4333", "28", "321", "8.72%", "456", "6.14%"],
    ["Pasadena, CA (Old Pasadena)", "34.1492, -118.1417", "2", "27", "7.41%", "70", "2.86%"],
    ["Los Angeles, CA (Huntington Park / South LA)", "33.9792, -118.2480", "2", "30", "6.67%", "141", "1.42%"],
    ["Augusta, GA (West Augusta / Martinez)", "33.4867, -82.0802", "4", "67", "5.97%", "127", "3.15%"],
    ["Manhattan, NY (Lower East Side / Chinatown)", "40.7159, -73.9966", "18", "308", "5.84%", "681", "2.64%"],
    ["Los Angeles, CA (Koreatown)", "34.0634, -118.2923", "8", "147", "5.44%", "318", "2.52%"],
    ["Los Angeles Area, CA (South Gate)", "33.9666, -118.2037", "2", "42", "4.76%", "93", "2.15%"],
    ["San Francisco, CA (West Portal)", "37.7306, -122.4754", "2", "45", "4.44%", "215", "0.93%"],
    ["Los Angeles, CA (USC / Historic South)", "34.0262, -118.2731", "4", "93", "4.30%", "296", "1.35%"],
    ["Los Angeles, CA (UCLA / Westwood)", "34.0652, -118.4422", "2", "60", "3.33%", "183", "1.09%"]
]

columns = [
    "Exact Location / Core Neighborhood",
    "Center Coordinates (Lat, Lon)",
    "Low-Rate Clinics Count",
    "Total Clinics (2 mi)",
    "Low-Rate Pct (2 mi)",
    "Total Clinics (5 mi)",
    "Low-Rate Pct (5 mi)"
]

df = pd.DataFrame(data, columns=columns)

print(df.to_string(index=False))

In [ ]:
df['rating_value'] = pd.to_numeric(df['rating_value'], errors='coerce')
df_valid = df.dropna(subset=['latitude', 'longitude', 'rating_value']).copy()

df_high = df_valid[df_valid['rating_value'] > 4.8].copy()

coords_high = np.radians(df_high[['latitude', 'longitude']].values)
coords_all = np.radians(df_valid[['latitude', 'longitude']].values)

db = DBSCAN(eps=2.0/3958.8, min_samples=2, metric='haversine').fit(coords_high)
df_high['cluster'] = db.labels_
clusters = df_high[df_high['cluster'] != -1]

results = []
tree_all = BallTree(coords_all, metric='haversine')
tree_high = BallTree(coords_high, metric='haversine')

for c_id, group in clusters.groupby('cluster'):
    center_lat = group['latitude'].mean()
    center_lon = group['longitude'].mean()

    centroid_rad = np.radians([[center_lat, center_lon]])

    n_total_2mi = tree_all.query_radius(centroid_rad, r=2.0/3958.8, count_only=True)[0]
    n_total_5mi = tree_all.query_radius(centroid_rad, r=5.0/3958.8, count_only=True)[0]

    n_high_2mi = tree_high.query_radius(centroid_rad, r=2.0/3958.8, count_only=True)[0]
    n_high_5mi = tree_high.query_radius(centroid_rad, r=5.0/3958.8, count_only=True)[0]

    if n_total_2mi < 10:
        continue

    high_pct_2mi = n_high_2mi / n_total_2mi if n_total_2mi > 0 else 0
    high_pct_5mi = n_high_5mi / n_total_5mi if n_total_5mi > 0 else 0

    results.append({
        'Cluster_ID': c_id,
        'Center_Lat': round(center_lat, 4),
        'Center_Lon': round(center_lon, 4),
        'High_Clinics_2mi': n_high_2mi,
        'Total_Clinics_2mi': n_total_2mi,
        'High_Pct_2mi': high_pct_2mi,
        'Total_Clinics_5mi': n_total_5mi,
        'High_Pct_5mi': high_pct_5mi
    })


res_df = pd.DataFrame(results).sort_values(by=['High_Pct_2mi', 'High_Clinics_2mi'], ascending=[False, False])


res_df['High_Pct_2mi'] = res_df['High_Pct_2mi'].apply(lambda x: f"{x:.2%}")
res_df['High_Pct_5mi'] = res_df['High_Pct_5mi'].apply(lambda x: f"{x:.2%}")

print(res_df.head(25).to_string(index=False))

In [ ]:
import pandas as pd

# Data corresponds to your provided clusters (Top-Rated Clinics defined as > 4.8)
# Sorted descending by High_Pct_2mi
data = [
    ["Torrance, CA (Del Amo / South Bay)", "33.8459, -118.3618", "10", "11", "90.91%", "37", "72.97%"],
    ["Sherman Oaks / Encino, CA", "34.1573, -118.4900", "9", "12", "75.00%", "29", "62.07%"],
    ["Long Beach / Lakewood, CA", "33.8050, -118.1242", "8", "11", "72.73%", "36", "66.67%"],
    ["Forsyth / Macon North, GA", "32.9040, -83.7063", "26", "37", "70.27%", "62", "61.29%"],
    ["Atlanta, GA (Buckhead / Piedmont)", "33.8341, -84.3659", "33", "51", "64.71%", "211", "60.66%"],
    ["Atlanta, GA (Vinings / Cumberland)", "33.8493, -84.4361", "20", "31", "64.52%", "131", "57.25%"],
    ["San Francisco, CA (Hayes Valley / Fillmore)", "37.7790, -122.4263", "197", "332", "59.34%", "456", "56.80%"],
    ["Modesto, CA (Downtown / McHenry)", "37.6736, -120.9936", "70", "120", "58.33%", "180", "52.22%"],
    ["Toccoa, GA", "34.5751, -83.3151", "7", "12", "58.33%", "17", "52.94%"],
    ["Buffalo, NY (South Buffalo / Lackawanna)", "42.8488, -78.7717", "26", "45", "57.78%", "83", "48.19%"],
    ["Manhattan, NY (East Village / Gramercy)", "40.7348, -73.9777", "190", "331", "57.40%", "718", "58.91%"],
    ["Amherst, NY (UB North Campus)", "42.9550, -78.8029", "21", "45", "46.67%", "195", "45.13%"],
    ["Vidalia, GA", "32.2066, -82.3897", "18", "39", "46.15%", "41", "43.90%"],
    ["Augusta, GA (Martinez / Evans)", "33.4886, -82.0573", "26", "63", "41.27%", "154", "42.21%"],
    ["Fort Bragg, CA", "39.4317, -123.8017", "9", "24", "37.50%", "24", "37.50%"],
    ["Saranac Lake, NY", "44.3320, -74.1324", "6", "16", "37.50%", "16", "37.50%"],
    ["Los Angeles, CA (Koreatown)", "34.0634, -118.2983", "49", "132", "37.12%", "312", "45.83%"],
    ["Macon, GA (Downtown)", "32.8415, -83.6418", "17", "47", "36.17%", "77", "36.36%"],
    ["Eureka, CA", "40.7892, -124.1508", "16", "66", "24.24%", "68", "23.53%"],
    ["Macon, GA (South / Eisenhower)", "32.8248, -83.7107", "3", "14", "21.43%", "80", "41.25%"],
    ["Brooklyn, NY (Flatbush / Crown Heights)", "40.6525, -73.9278", "2", "13", "15.38%", "163", "49.08%"]
]

columns = [
    "Exact Location / Core Neighborhood",
    "Center Coordinates (Lat, Lon)",
    "Top-Rate Clinics Count (>4.8 within 2 mi)",
    "Total Clinics (2 mi)",
    "Top-Rate Pct (2 mi)",
    "Total Clinics (5 mi)",
    "Top-Rate Pct (5 mi)"
]

df_top_rated = pd.DataFrame(data, columns=columns)

print(df_top_rated.to_string(index=False))

=========To hub==========
* 90502: Torrance, CA
* 30474: Vidalia, GA (small city)
* 30324: Atlanta, GA (Buckhead/Lindbergh area)
* 90001: Los Angeles, CA (South LA/Florence)
* 30331: Atlanta, GA (Southwest Atlanta)

=========To peer=========
* 14226: Amherst/Williamsville, NY (Buffalo suburb)
* 31201: Macon, GA (Downtown)
* 14203: Buffalo, NY (Downtown)
* 10038: New York, NY (Manhattan - Financial District)
* 10007: New York, NY (Manhattan - Tribeca/Civic Center)


## Regression

### Panel

In [ ]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import statsmodels.formula.api as smf
from sklearn.neighbors import BallTree
from sklearn.metrics import DistanceMetric
import warnings
warnings.filterwarnings('ignore')

print("Build Panel Data")

df_reg = df.dropna(subset=['rating_value', 'latitude', 'longitude', 'start_date', 'mapped_location']).copy()
df_reg['entry_year'] = pd.to_datetime(df_reg['start_date'], errors='coerce').dt.year
df_reg = df_reg.dropna(subset=['entry_year'])
df_reg['entry_year'] = df_reg['entry_year'].astype(int)

df_reg = df_reg.reset_index(drop=True)
df_reg['clinic_id'] = ["C_" + str(i) for i in range(len(df_reg))]

# strong competitor
df_reg['city_avg_votes'] = df_reg.groupby('mapped_location')['votes_count'].transform('mean')
df_reg['is_strong'] = (df_reg['rating_value'] > 4.9) & (df_reg['votes_count'] > df_reg['city_avg_votes'])

# 25% and 50% of the distance between clinics within each city
EARTH_RADIUS = 3958.8
dist = DistanceMetric.get_metric('haversine')
city_thresholds = {}

for city, group in df_reg.groupby('mapped_location'):
    coords = np.radians(group[['latitude', 'longitude']].values)
    if len(coords) > 1:
        dist_matrix = dist.pairwise(coords) * EARTH_RADIUS
        triu_indices = np.triu_indices(len(coords), k=1)
        pairwise_dists = dist_matrix[triu_indices]
        p25 = np.percentile(pairwise_dists, 25) / EARTH_RADIUS
        p50 = np.percentile(pairwise_dists, 50) / EARTH_RADIUS
    else:
        p25, p50 = 0, 0
    city_thresholds[city] = {'p25': p25, 'p50': p50}

coords_rad = np.radians(df_reg[['latitude', 'longitude']].values)
tree = BallTree(coords_rad, metric='haversine')

# For each clinic get the neighbors within 25% and 50% radius.
indices_25pct = []
indices_50pct = []
for i, row in df_reg.iterrows():
    city = row['mapped_location']
    r_25 = city_thresholds.get(city, {'p25': 0})['p25']
    r_50 = city_thresholds.get(city, {'p50': 0})['p50']

    pt = coords_rad[i:i+1]
    indices_25pct.append(tree.query_radius(pt, r=r_25)[0])
    indices_50pct.append(tree.query_radius(pt, r=r_50)[0])

panel_records = []
for i, row in df_reg.iterrows():
    entry_y = int(row['entry_year'])

    neighbors_25 = indices_25pct[i]
    years_25 = df_reg.iloc[neighbors_25]['entry_year'].values
    titles_25 = df_reg.iloc[neighbors_25]['title'].values
    ids_25 = df_reg.iloc[neighbors_25]['clinic_id'].values
    strong_25 = df_reg.iloc[neighbors_25]['is_strong'].values

    neighbors_50 = indices_50pct[i]
    years_50 = df_reg.iloc[neighbors_50]['entry_year'].values
    titles_50 = df_reg.iloc[neighbors_50]['title'].values
    ids_50 = df_reg.iloc[neighbors_50]['clinic_id'].values
    strong_50 = df_reg.iloc[neighbors_50]['is_strong'].values

    for current_year in range(entry_y, 2026):
        # 25%
        density_25 = max(0, np.sum(years_25 <= current_year) - 1)
        lag_25 = np.sum(years_25 == (current_year - 1))
        shock_mask_25 = (years_25 == (current_year - 1)) & (ids_25 != row['clinic_id'])
        strong_lag_25 = np.sum((years_25 == (current_year - 1)) & strong_25 & (ids_25 != row['clinic_id']))
        names_25 = ", ".join(titles_25[shock_mask_25])

        # 50%
        density_50 = max(0, np.sum(years_50 <= current_year) - 1)
        lag_50 = np.sum(years_50 == (current_year - 1))
        shock_mask_50 = (years_50 == (current_year - 1)) & (ids_50 != row['clinic_id'])
        strong_lag_50 = np.sum((years_50 == (current_year - 1)) & strong_50 & (ids_50 != row['clinic_id']))
        names_50 = ", ".join(titles_50[shock_mask_50])

        # 25%-50%
        density_50_ring = max(0, np.sum(years_50 <= current_year) - 1) - density_25
        lag_50_ring = np.sum(years_50 == (current_year - 1)) - lag_25
        shock_mask_50_ring = (years_50 == (current_year - 1)) & (ids_50 != row['clinic_id']) & ~np.isin(ids_50, ids_25)
        strong_lag_50_ring = strong_lag_50 - strong_lag_25
        names_50_ring = ", ".join(titles_50[shock_mask_50_ring])

        panel_records.append({
            'clinic_id': row['clinic_id'],
            'title': row['title'],
            'zip_clean': row['zip_clean'],
            'year': current_year,
            'clinic_age': current_year - entry_y,

            'log_density_25pct_total': np.log1p(density_25),
            'lag_entry_shock_25pct_count': lag_25,
            'lag_entry_shock_25pct_names': names_25,
            'lag_entry_shock_25pct_dummy': 1 if lag_25 > 0 else 0,
            'log_lag_entry_shock_25pct': np.log1p(lag_25),
            'strong_entry_shock_25pct_count': strong_lag_25,
            'log_strong_entry_shock_25pct': np.log1p(strong_lag_25),

            'log_density_50pct_total': np.log1p(density_50),
            'lag_entry_shock_50pct_count': lag_50,
            'lag_entry_shock_50pct_names': names_50,
            'lag_entry_shock_50pct_dummy': 1 if lag_50 > 0 else 0,
            'log_lag_entry_shock_50pct': np.log1p(lag_50),
            'strong_entry_shock_50pct_count': strong_lag_50,
            'log_strong_entry_shock_50pct': np.log1p(strong_lag_50),

            'log_density_25_to_50pct_total': np.log1p(density_50_ring),
            'lag_entry_shock_25_to_50pct_count': lag_50_ring,
            'lag_entry_shock_25_to_50pct_names': names_50_ring,
            'lag_entry_shock_25_to_50pct_dummy': 1 if lag_50_ring > 0 else 0,
            'log_lag_entry_shock_25_to_50pct': np.log1p(lag_50_ring),
            'strong_entry_shock_25_to_50pct_count': strong_lag_50_ring,
            'log_strong_entry_shock_25_to_50pct': np.log1p(strong_lag_50_ring),

            'distance_to_hub': row['distance_to_hub'],
            'mapped_location': row['mapped_location'],
            'Keyword_Category': row['Keyword_Category']
        })

df_panel = pd.DataFrame(panel_records)

# Process all review for dynamic ratings and votes
REVIEW_NAME_COL = 'shop_title'
REVIEW_ZIP_COL = 'shop_zip'
REVIEW_DATE_COL = 'date'
REVIEW_RATING_COL = 'rating'

chunk_size = 10000
yearly_stats_chunks = []
cols_to_use = [REVIEW_NAME_COL, REVIEW_ZIP_COL, REVIEW_DATE_COL, REVIEW_RATING_COL]

print("Chunking and aggregating reviews...")
try:
    for chunk in pd.read_csv("/content/drive/MyDrive/RA/health_care/dentist_LMS_keywords/output/final/all_reviews_detailed.csv", chunksize=chunk_size, usecols=cols_to_use):
        # match zip code
        chunk[REVIEW_ZIP_COL] = chunk[REVIEW_ZIP_COL].astype(str).str.replace(r'\.0$', '', regex=True).str.strip()
        chunk['review_year'] = pd.to_datetime(chunk[REVIEW_DATE_COL], errors='coerce').dt.year
        chunk = chunk.dropna(subset=['review_year', REVIEW_RATING_COL])
        chunk['review_year'] = chunk['review_year'].astype(int)

        chunk['is_bad_review'] = (chunk[REVIEW_RATING_COL] <= 3).astype(int)

        # aggregate count and sum per year
        stats_df = chunk.groupby([REVIEW_NAME_COL, REVIEW_ZIP_COL, 'review_year']).agg(
            new_count=(REVIEW_RATING_COL, 'count'),
            new_stars=(REVIEW_RATING_COL, 'sum'),
            bad_count=('is_bad_review', 'sum')
        ).reset_index()
        yearly_stats_chunks.append(stats_df)

    final_yearly_stats = pd.concat(yearly_stats_chunks).groupby([REVIEW_NAME_COL, REVIEW_ZIP_COL, 'review_year']).sum().reset_index()
    final_yearly_stats = final_yearly_stats.rename(columns={REVIEW_NAME_COL: 'title', REVIEW_ZIP_COL: 'zip_str'})
except Exception as e:
    print(f"   [Warning] Failed to read reviews: {e}")

# Merge reviews into panel
df_panel['zip_str'] = df_panel['zip_clean'].astype(str).str.replace(r'\.0$', '', regex=True).str.strip()

df_panel = df_panel.merge(final_yearly_stats, left_on=['title', 'zip_str', 'year'], right_on=['title', 'zip_str', 'review_year'], how='left')
df_panel['new_count'] = df_panel['new_count'].fillna(0)
df_panel['new_stars'] = df_panel['new_stars'].fillna(0)
df_panel['bad_count'] = df_panel['bad_count'].fillna(0)

# sort
df_panel = df_panel.sort_values(by=['clinic_id', 'year'])
df_panel['cumulative_votes'] = df_panel.groupby('clinic_id')['new_count'].cumsum()
df_panel['cumulative_stars'] = df_panel.groupby('clinic_id')['new_stars'].cumsum()
df_panel['cumulative_bad_votes'] = df_panel.groupby('clinic_id')['bad_count'].cumsum()

# dynamic vote
df_panel['dynamic_rating'] = np.where(df_panel['cumulative_votes'] > 0, df_panel['cumulative_stars'] / df_panel['cumulative_votes'], np.nan)
df_panel['log_votes_dynamic'] = np.log1p(df_panel['cumulative_votes'])


df_panel['current_bad_review_pct'] = np.where(df_panel['new_count'] > 0, df_panel['bad_count'] / df_panel['new_count'], np.nan)


df_panel['cumulative_bad_review_pct'] = np.where(df_panel['cumulative_votes'] > 0, df_panel['cumulative_bad_votes'] / df_panel['cumulative_votes'], np.nan)


# drop early years where the clinic hadn't received first review
df_panel_valid = df_panel.dropna(subset=['dynamic_rating']).reset_index(drop=True)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
print("number of clinics got review")
yearly_counts = df_panel_valid['review_year'].value_counts().sort_index()
print(yearly_counts)


In [ ]:
yearly_review_volume = df_panel_valid.groupby('year')['new_count'].sum().astype(int)

print("number of reviews")
print(yearly_review_volume)

In [ ]:
yearly_entries = df_reg['entry_year'].value_counts().sort_index().astype(int)

print("number of new clinics")
print(yearly_entries)

In [ ]:
import pandas as pd
import numpy as np

print("Panel Data Check")

print("\nData Info")
print(f"Observations: {len(df_panel_valid)}")
print(f"Unique clinic(N): {df_panel_valid['clinic_id'].nunique()}")
print(f"Time range(T): {df_panel_valid['year'].min()} to {df_panel_valid['year'].max()}")

# Check missing data
print("\nCheck missing data")
core_vars = ['dynamic_rating', 'cumulative_votes', 'log_votes_dynamic', 'lag_entry_shock_25pct_count', 'log_density_25pct_total', 'lag_entry_shock_50pct_count', 'log_density_50pct_total']
print(df_panel_valid[core_vars].isnull().sum())

# Stat
print("\nStat")
print(df_panel_valid[core_vars].describe().round(3))



In [ ]:
print("Panel Data Check")

print("\nData Info")
print(f"Observations: {len(df_panel_valid)}")
print(f"Unique clinic(N): {df_panel_valid['clinic_id'].nunique()}")
print(f"Time range(T): {df_panel_valid['year'].min()} to {df_panel_valid['year'].max()}")

# Check missing data
print("\nCheck missing data")
core_vars = ['dynamic_rating', 'cumulative_votes', 'log_votes_dynamic', 'lag_entry_shock_25pct_count', 'log_density_25pct_total', 'lag_entry_shock_50pct_count', 'log_density_50pct_total']
print(df_panel_valid[core_vars].isnull().sum())

# Stat by Year
print("\nStat by Year")
# Loop through each year in chronological order
for year in sorted(df_panel_valid['year'].unique()):
    print(f"\n{'-'*50}")
    print(f"Summary Statistics for Year: {year}")
    print(f"{'-'*50}")
    # Filter data for the specific year and print the describe() table
    year_data = df_panel_valid[df_panel_valid['year'] == year][core_vars]
    print(year_data.describe().round(3))

In [ ]:
# random clinic
print("\nRandom clinic info")
sample_clinic = np.random.choice(df_panel_valid['clinic_id'].unique())
full_sample_data = df_panel_valid[df_panel_valid['clinic_id'] == sample_clinic]

clinic_title = full_sample_data['title'].iloc[0]
clinic_zip = full_sample_data['zip_clean'].iloc[0]

sample_data = df_panel_valid[df_panel_valid['clinic_id'] == sample_clinic][
    ['year', 'new_count', 'cumulative_votes', 'dynamic_rating',
     'lag_entry_shock_25pct_count', 'log_density_25pct_total', 'lag_entry_shock_25pct_names',
     'lag_entry_shock_50pct_count', 'log_density_50pct_total', 'lag_entry_shock_50pct_names']
]
print(f"Sample clinic: {sample_clinic}")
print(f"Clinic Name: {clinic_title} (Zip: {clinic_zip})")
print(sample_data.to_string(index=False))

In [ ]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import statsmodels.formula.api as smf

# Lag Entry Shock Comparison
print("\nLag Entry Shock Comparison")

# Using the Geographic FE framework as it optimally balances macro controls and time dynamics
formula_comp_25 = "dynamic_rating ~ {} + log_density_25pct_total + clinic_age + log_votes_dynamic + distance_to_hub + C(mapped_location)"
formula_comp_50 = "dynamic_rating ~ {} + log_density_50pct_total + clinic_age + log_votes_dynamic + distance_to_hub + C(mapped_location)"

models_to_run = [
    ('25% Dummy (0/1)', formula_comp_25, 'lag_entry_shock_25pct_dummy'),
    ('25% Cont. Count', formula_comp_25, 'lag_entry_shock_25pct_count'),
    ('25% Log(Count)',  formula_comp_25, 'log_lag_entry_shock_25pct'),
    ('50% Dummy (0/1)', formula_comp_50, 'lag_entry_shock_50pct_dummy'),
    ('50% Cont. Count', formula_comp_50, 'lag_entry_shock_50pct_count'),
    ('50% Log(Count)',  formula_comp_50, 'log_lag_entry_shock_50pct')
]

results = []
for name, formula, var in models_to_run:
    mod = smf.ols(formula.format(var), data=df_panel_valid).fit()
    results.append({
        'Shock Form': name,
        'AIC': round(mod.aic, 2),
        'BIC': round(mod.bic, 2),
        'R-squared': round(mod.rsquared, 4),
        'Coefficient': round(mod.params[var], 5),
        'P-value': f"{mod.pvalues[var]:.3e}"
    })

comparison_df = pd.DataFrame(results)
print(comparison_df.to_string(index=False))


In [ ]:
# Geographic fixed effect
print("Geographic fixed effect")

formula_geo_pooled = "dynamic_rating ~ log_lag_entry_shock_25pct + log_density_25pct_total + clinic_age + log_votes_dynamic + distance_to_hub"
mod_geo_pooled = smf.ols(formula_geo_pooled, data=df_panel_valid).fit()

# F-Test
mod_geo_fe = smf.ols(formula_geo_pooled + " + C(mapped_location)", data=df_panel_valid).fit()
f_test_geo = mod_geo_fe.compare_f_test(mod_geo_pooled)
print(f"[Geo F-Test] P-value: {f_test_geo[1]:.4e} -> {'Reject the null hypothesis. Significant geographic fixed effects exist.' if f_test_geo[1]<0.05 else 'Cannot reject the null hypothesis. Geographic fixed effects not significant.'}")

# Hausman Test
try:
    mod_geo_re = smf.mixedlm(formula_geo_pooled, data=df_panel_valid, groups=df_panel_valid['mapped_location']).fit()
    cv_geo = [v for v in mod_geo_re.params.index if v in mod_geo_fe.params.index and v != 'Intercept']
    diff_geo = mod_geo_fe.params[cv_geo] - mod_geo_re.params[cv_geo]
    v_diff_geo = mod_geo_fe.cov_params().loc[cv_geo, cv_geo] - mod_geo_re.cov_params().loc[cv_geo, cv_geo]

    try:
        chi2_geo = np.dot(diff_geo.T, np.dot(np.linalg.inv(v_diff_geo), diff_geo))
        if chi2_geo < 0:
            print(f"[Geo Hausman] Chi2: {chi2_geo:.4f} (Invalid: Non-positive definite matrix. )")
        else:
            pval_geo = 1 - stats.chi2.cdf(chi2_geo, len(cv_geo))
            print(f"[Geo Hausman] Chi2: {chi2_geo:.4f}, P-value: {pval_geo:.4e} -> {'Use FE' if pval_geo<0.05 else 'Use RE'}")
    except np.linalg.LinAlgError:
        print("[Geo Hausman Warning] Singular matrix detected. Attempting pseudo-inverse...")
        chi2_geo_pinv = np.dot(diff_geo.T, np.dot(np.linalg.pinv(v_diff_geo), diff_geo))
        if chi2_geo_pinv < 0:
            print(f"[Geo Hausman Pseudo-Inv] Chi2: {chi2_geo_pinv:.4f} (Invalid: Non-positive definite. )")
        else:
            pval_geo_pinv = 1 - stats.chi2.cdf(chi2_geo_pinv, len(cv_geo))
            print(f"[Geo Hausman Pseudo-Inv] Chi2: {chi2_geo_pinv:.4f}, P-value: {pval_geo_pinv:.4e} -> {'Use FE' if pval_geo_pinv<0.05 else 'Use RE'}")
except Exception as e:
    print(f"[Geo Hausman Error] Computation failed: {e}")



# Individual Fixed Effect Tests
print("\nIndividual fixed effect")
print("'distance_to_hub' and 'mapped_location' are excluded")

formula_ind_pooled = "dynamic_rating ~ log_lag_entry_shock_25pct + log_density_25pct_total + clinic_age + log_votes_dynamic"
mod_ind_pooled = smf.ols(formula_ind_pooled, data=df_panel_valid).fit()

# F-Test
try:
    mod_ind_fe = smf.ols(formula_ind_pooled + " + C(clinic_id)", data=df_panel_valid).fit()
    f_test_ind = mod_ind_fe.compare_f_test(mod_ind_pooled)
    print(f"[Ind F-Test] P-value: {f_test_ind[1]:.4e} -> {'Reject the null hypothesis. Significant individual fixed effects exist.' if f_test_ind[1]<0.05 else 'Cannot reject the null hypothesis. Individual fixed effects not significant.'}")
except Exception as e:
    print(f"[Ind F-Test] Computation failed: {e}")

# Hausman Test
try:
    mod_ind_re = smf.mixedlm(formula_ind_pooled, data=df_panel_valid, groups=df_panel_valid['clinic_id']).fit()
    cv_ind = [v for v in mod_ind_re.params.index if v in mod_ind_fe.params.index and v != 'Intercept']
    diff_ind = mod_ind_fe.params[cv_ind] - mod_ind_re.params[cv_ind]
    v_diff_ind = mod_ind_fe.cov_params().loc[cv_ind, cv_ind] - mod_ind_re.cov_params().loc[cv_ind, cv_ind]

    try:
        chi2_ind = np.dot(diff_ind.T, np.dot(np.linalg.inv(v_diff_ind), diff_ind))
        if chi2_ind < 0:
            print(f"[Ind Hausman] Chi2: {chi2_ind:.4f} (Invalid: Non-positive definite matrix)")
        else:
            pval_ind = 1 - stats.chi2.cdf(chi2_ind, len(cv_ind))
            print(f"[Ind Hausman] Chi2: {chi2_ind:.4f}, P-value: {pval_ind:.4e} -> {'use FE' if pval_ind<0.05 else 'use RE'}")
    except np.linalg.LinAlgError:
        print("[Ind Hausman Warning] Singular matrix detected. Attempting pseudo-inverse...")
        chi2_ind_pinv = np.dot(diff_ind.T, np.dot(np.linalg.pinv(v_diff_ind), diff_ind))
        if chi2_ind_pinv < 0:
            print(f"[Ind Hausman Pseudo-Inv] Chi2: {chi2_ind_pinv:.4f} (Invalid: Non-positive definite)")
        else:
            pval_ind_pinv = 1 - stats.chi2.cdf(chi2_ind_pinv, len(cv_ind))
            print(f"[Ind Hausman Pseudo-Inv] Chi2: {chi2_ind_pinv:.4f}, P-value: {pval_ind_pinv:.4e} -> {'use FE' if pval_ind_pinv<0.05 else 'use RE'}")
except Exception as e:
    print(f"[Ind Hausman Error] Computation or convergence failed: {e}")


In [ ]:
!pip install linearmodels

In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf
from linearmodels.panel import PanelOLS
import numpy as np

df_panel_valid['other_entry_shock_25pct_count'] = df_panel_valid['lag_entry_shock_25pct_count'] - df_panel_valid['strong_entry_shock_25pct_count']
df_panel_valid['log_other_entry_shock_25pct'] = np.log1p(df_panel_valid['other_entry_shock_25pct_count'])

df_panel_valid['other_entry_shock_25_to_50pct_count'] = df_panel_valid['lag_entry_shock_25_to_50pct_count'] - df_panel_valid['strong_entry_shock_25_to_50pct_count']
df_panel_valid['log_other_entry_shock_25_to_50pct'] = np.log1p(df_panel_valid['other_entry_shock_25_to_50pct_count'])

df_panel_valid['other_entry_shock_50pct_count'] = df_panel_valid['lag_entry_shock_50pct_count'] - df_panel_valid['strong_entry_shock_50pct_count']
df_panel_valid['log_other_entry_shock_50pct'] = np.log1p(df_panel_valid['other_entry_shock_50pct_count'])





In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

shock_configs = [
    ('strong_entry_shock_25pct_count', '0-25%'),
    ('strong_entry_shock_25_to_50pct_count', '25-50%'),
    ('strong_entry_shock_50pct_count', '0-50%')
]

sns.set_theme(style="white")
palette = {'Before Strong Entry': '#3498db', 'After Strong Entry': '#e74c3c'}

for shock_col, label in shock_configs:
    if isinstance(df_panel_valid.index, pd.MultiIndex):
        df_plot_data = df_panel_valid.reset_index()
    else:
        df_plot_data = df_panel_valid.copy()

    df_plot_data['post_strong_entry'] = (df_plot_data.groupby('clinic_id')[shock_col].cumsum() > 0).astype(int)

    clinics_with_shock = df_plot_data[df_plot_data['post_strong_entry'] == 1]['clinic_id'].unique()
    df_plot_data = df_plot_data[df_plot_data['clinic_id'].isin(clinics_with_shock)].copy()

    if df_plot_data.empty:
        continue

    df_plot_data['Comp_Group'] = df_plot_data['post_strong_entry'].map({
        0: 'Before Strong Entry',
        1: 'After Strong Entry'
    })

    mean_before = df_plot_data[df_plot_data['Comp_Group'] == 'Before Strong Entry']['dynamic_rating'].mean()
    mean_after = df_plot_data[df_plot_data['Comp_Group'] == 'After Strong Entry']['dynamic_rating'].mean()

    fig_kde, ax_kde = plt.subplots(figsize=(10, 6), dpi=100)
    sns.kdeplot(data=df_plot_data, x="dynamic_rating", hue="Comp_Group",
                fill=True, alpha=0.5, bw_adjust=1.2, palette=palette,
                lw=2, ax=ax_kde, common_norm=False)

    ax_kde.axvline(mean_before, color='#2980b9', linestyle='--', lw=2)
    ax_kde.text(mean_before*0.98, ax_kde.get_ylim()[1]*0.85, f'Mean Before: {mean_before:.2f}',
                fontsize=11, fontweight='bold', color='#2980b9', ha='right')

    ax_kde.axvline(mean_after, color='#c0392b', linestyle='--', lw=2)
    ax_kde.text(mean_after*1.02, ax_kde.get_ylim()[1]*0.75, f'Mean After: {mean_after:.2f}',
                fontsize=11, fontweight='bold', color='#c0392b', ha='left')

    ax_kde.axvline(3.5, color='black', linestyle='-', lw=2)
    ax_kde.text(3.5*0.98, ax_kde.get_ylim()[1]*0.5, 'Cutoff: 3.5',
                fontsize=11, fontweight='bold', color='black', ha='right')

    ax_kde.set_xlim(2.5, 5.1)
    ax_kde.set_xlabel("Dynamic Rating", fontsize=12)
    ax_kde.set_ylabel("Density", fontsize=12)
    ax_kde.set_title(f"Density Shift: Before vs. After Strong Entry ({label})", fontsize=15, fontweight='bold', y=1.05)
    sns.despine(left=True)
    plt.tight_layout()
    plt.show()

    fig_hist, ax_hist = plt.subplots(figsize=(10, 6), dpi=100)
    sns.histplot(data=df_plot_data, x="dynamic_rating", hue="Comp_Group",
                 palette=palette, edgecolor='white', bins=15, alpha=0.6,
                 stat='density', ax=ax_hist, common_norm=False, multiple="layer")

    ax_hist.axvline(mean_before, color='#2980b9', linestyle='--', lw=2)
    ax_hist.text(mean_before*0.98, ax_hist.get_ylim()[1]*0.85, f'Mean Before: {mean_before:.2f}',
                 fontsize=11, fontweight='bold', color='#2980b9', ha='right')

    ax_hist.axvline(mean_after, color='#c0392b', linestyle='--', lw=2)
    ax_hist.text(mean_after*1.02, ax_hist.get_ylim()[1]*0.75, f'Mean After: {mean_after:.2f}',
                 fontsize=11, fontweight='bold', color='#c0392b', ha='left')

    ax_hist.axvline(3.5, color='black', linestyle='-', lw=2)

    ax_hist.set_xlim(2.5, 5.1)
    ax_hist.set_xlabel("Dynamic Rating", fontsize=12)
    ax_hist.set_ylabel("Density", fontsize=12)
    ax_hist.set_title(f"Histogram Shift: Before vs. After Strong Entry ({label})", fontsize=15, fontweight='bold', y=1.05)
    sns.despine(left=True)
    plt.tight_layout()
    plt.show()

In [ ]:
shock_configs = [
    ('strong_entry_shock_25pct_count', '0-25%'),
    ('strong_entry_shock_25_to_50pct_count', '25-50%'),
    ('strong_entry_shock_50pct_count', '0-50%')
]

bins = [1.0, 2.0, 3.0, 4.0, 5.0]
labels = ["1-2", "2-3",  "3-4",  "4-5"]

for shock_col, label in shock_configs:
    print(f"\nShock: {label} ({shock_col})")

    if isinstance(df_panel_valid.index, pd.MultiIndex):
        df_temp = df_panel_valid.reset_index()
    else:
        df_temp = df_panel_valid.copy()

    df_temp['post_strong_entry'] = (df_temp.groupby('clinic_id')[shock_col].cumsum() > 0).astype(int)

    clinics_with_shock = df_temp[df_temp['post_strong_entry'] == 1]['clinic_id'].unique()
    df_temp = df_temp[df_temp['clinic_id'].isin(clinics_with_shock)].copy()

    if df_temp.empty:
        print("No data.")
        continue

    df_temp['Comp_Group'] = df_temp['post_strong_entry'].map({
        0: 'Before Strong Entry',
        1: 'After Strong Entry'
    })

    for group in df_temp['Comp_Group'].unique():
        group_data = df_temp[df_temp['Comp_Group'] == group]['dynamic_rating']
        count = pd.cut(group_data, bins=bins, labels=labels).value_counts()
        percent = (count / len(group_data)) * 100
        for interval, percentage in percent.sort_index().items():
            print(f"{group} {interval}: {percentage:.2f}%")


#LLM

In [ ]:
!pip install transformers torch tqdm pandas

In [ ]:
!pip install linearmodels

In [ ]:
import pandas as pd
import numpy as np
import torch
from transformers import pipeline
from tqdm.auto import tqdm

device = 0 if torch.cuda.is_available() else -1
print(f"Using device: {'GPU' if device == 0 else 'CPU'}")

classifier = pipeline("zero-shot-classification",
                      model="facebook/bart-large-mnli",
                      device=device)

df_reviews = pd.read_csv("/content/drive/MyDrive/RA/health_care/dentist_LMS_keywords/output/final/all_reviews_detailed.csv")
df_reviews = df_reviews.dropna(subset=['review_text', 'timestamp', 'shop_title'])
df_reviews['review_year'] = pd.to_datetime(df_reviews['timestamp']).dt.year
len(df_reviews)

In [ ]:
import os
import pandas as pd
import numpy as np
from tqdm.auto import tqdm

INTERMEDIATE_FILE = "/content/drive/MyDrive/RA/health_care/dentist_LMS_keywords/temp/NLP/nlp_intermediate_results.csv"
FINAL_FILE = "/content/drive/MyDrive/RA/health_care/dentist_LMS_keywords/output/final/clinic_yearly_nlp_traits.csv"

os.makedirs(os.path.dirname(INTERMEDIATE_FILE), exist_ok=True)
os.makedirs(os.path.dirname(FINAL_FILE), exist_ok=True)

# label(demo)
candidate_labels = [
    "advertisement, coupon, promotion, or special discount", # ad
    "pediatric dentistry, kids, or elderly care",            # age
    "cosmetic dentistry, teeth whitening, or veneers",       # 美容
    "orthodontics, braces, or Invisalign",                   # 矫正
    "oral surgery, dental implants, or tooth extraction",    # 手术
    "high price, expensive, hidden fees, or cost issues",    # price
    "long wait time, rude staff, or bad customer service",   # service
    "comparing to another dentist or previous clinic"        # other clinics
]

THRESHOLD = 0.6
batch_size = 32
CHUNK_SIZE = 10000

if os.path.exists(INTERMEDIATE_FILE):
    processed_df = pd.read_csv(INTERMEDIATE_FILE, usecols=['shop_title'])
    start_idx = len(processed_df)
    print(f"Found existing records. Resuming from row {start_idx} / {len(df_reviews)}...")
else:
    start_idx = 0
    print(f"Total rows to process: {len(df_reviews)}")

for chunk_start in range(start_idx, len(df_reviews), CHUNK_SIZE):
    chunk_end = min(chunk_start + CHUNK_SIZE, len(df_reviews))
    print(f"\n>>> Processing rows {chunk_start} to {chunk_end}...")

    chunk_df = df_reviews.iloc[chunk_start:chunk_end].copy()
    texts = chunk_df['review_text'].astype(str).apply(lambda x: x[:512]).tolist()

    chunk_results = []

    for out in tqdm(classifier(texts, candidate_labels, multi_label=True, batch_size=batch_size), total=len(texts)):
        scores = dict(zip(out['labels'], out['scores']))
        chunk_results.append(scores)

    chunk_df['is_ad_driven'] = [1 if r["advertisement, coupon, promotion, or special discount"] > THRESHOLD else 0 for r in chunk_results]
    chunk_df['target_demographic'] = [1 if r["pediatric dentistry, kids, or elderly care"] > THRESHOLD else 0 for r in chunk_results]
    chunk_df['service_cosmetic'] = [1 if r["cosmetic dentistry, teeth whitening, or veneers"] > THRESHOLD else 0 for r in chunk_results]
    chunk_df['service_ortho'] = [1 if r["orthodontics, braces, or Invisalign"] > THRESHOLD else 0 for r in chunk_results]
    chunk_df['service_surgery'] = [1 if r["oral surgery, dental implants, or tooth extraction"] > THRESHOLD else 0 for r in chunk_results]
    chunk_df['issue_price'] = [1 if r["high price, expensive, hidden fees, or cost issues"] > THRESHOLD else 0 for r in chunk_results]
    chunk_df['issue_service'] = [1 if r["long wait time, rude staff, or bad customer service"] > THRESHOLD else 0 for r in chunk_results]
    chunk_df['is_comparing'] = [1 if r["comparing to another dentist or previous clinic"] > THRESHOLD else 0 for r in chunk_results]

    chunk_df.to_csv(INTERMEDIATE_FILE, mode='a', header=not os.path.exists(INTERMEDIATE_FILE), index=False)
    print(f"Chunk {chunk_start} to {chunk_end} saved")

print("\nAll processed. Loading full data for aggregation")
df_full_processed = pd.read_csv(INTERMEDIATE_FILE)

df_full_processed['shop_zip'] = df_full_processed['shop_zip'].astype(str).str.replace(r'\.0$', '', regex=True).str.strip()

df_clinic_traits = df_full_processed.groupby(['shop_title', 'shop_zip', 'review_year']).agg(
    total_reviews=('rating', 'count'),
    ad_ratio=('is_ad_driven', 'mean'),
    demo_ratio=('target_demographic', 'mean'),
    cosmetic_ratio=('service_cosmetic', 'mean'),
    ortho_ratio=('service_ortho', 'mean'),
    surgery_ratio=('service_surgery', 'mean'),
    price_issue_ratio=('issue_price', 'mean'),
    service_issue_ratio=('issue_service', 'mean'),
    compare_ratio=('is_comparing', 'mean')
).reset_index()

# review# * ad ratio
df_clinic_traits['ad_expenditure_proxy'] = df_clinic_traits['total_reviews'] * df_clinic_traits['ad_ratio']
df_clinic_traits['log_ad_expenditure'] = np.log1p(df_clinic_traits['ad_expenditure_proxy'])
df_clinic_traits = df_clinic_traits.rename(columns={
    'shop_title': 'title',
    'shop_zip': 'zip_str',
    'review_year': 'year'
})

df_clinic_traits.to_csv(FINAL_FILE, index=False)
print("Feature extraction saved")

In [ ]:
import pandas as pd

print("Merging NLP traits into df_panel_valid")

NLP_FILE = "/content/drive/MyDrive/RA/health_care/dentist_LMS_keywords/output/final/clinic_yearly_nlp_traits.csv"
df_nlp = pd.read_csv(NLP_FILE)

df_panel_valid = df_panel_valid.merge(
    df_nlp,
    on=['title', 'zip_str', 'year'],
    how='left'
)

nlp_cols = [
    'ad_ratio', 'demo_ratio', 'cosmetic_ratio', 'ortho_ratio',
    'surgery_ratio', 'price_issue_ratio', 'service_issue_ratio',
    'compare_ratio', 'log_ad_expenditure'
]

for col in nlp_cols:
    if col in df_panel_valid.columns:
        df_panel_valid[col] = df_panel_valid[col].fillna(0)

df_panel_valid = df_panel_valid.sort_values(by=['clinic_id', 'year'])
for col in nlp_cols:
    if col in df_panel_valid.columns:
        df_panel_valid[f'lag_{col}'] = df_panel_valid.groupby('clinic_id')[col].shift(1)
        df_panel_valid[f'lag_{col}'] = df_panel_valid[f'lag_{col}'].fillna(0)

df_first_bad = df_panel_valid[df_panel_valid['cumulative_bad_votes'] > 0].copy()
df_first_bad = df_first_bad.sort_values(by=['clinic_id', 'year']).groupby('clinic_id').first().reset_index()


PANEL_OUTPUT_PATH = "/content/drive/MyDrive/RA/health_care/dentist_LMS_keywords/output/final/panel_data_with_nlp.csv"
CROSS_OUTPUT_PATH = "/content/drive/MyDrive/RA/health_care/dentist_LMS_keywords/output/final/first_bad_with_nlp.csv"

df_panel_valid.to_csv(PANEL_OUTPUT_PATH, index=False)
df_first_bad.to_csv(CROSS_OUTPUT_PATH, index=False)

print(f"Panel Data saved to: {PANEL_OUTPUT_PATH} (Rows: {len(df_panel_valid)})")
print(f"Bad Review Data saved to: {CROSS_OUTPUT_PATH} (Rows: {len(df_first_bad)})")

In [ ]:
nlp_ratios = [
    'ad_ratio',
    'demo_ratio',
    'cosmetic_ratio',
    'ortho_ratio',
    'surgery_ratio',
    'price_issue_ratio',
    'service_issue_ratio',
    'compare_ratio'
]


total_reviews_in_panel = df_panel_valid['new_count'].sum()

for col in nlp_ratios:
    if col in df_panel_valid.columns:
        total_mentions = (df_panel_valid[col] * df_panel_valid['new_count']).sum()
        overall_pct = (total_mentions / total_reviews_in_panel) * 100
        print(f"{col:<20} : {overall_pct:>5.2f}%")

In [ ]:
print("\nLoading review-level NLP data for text sampling...")

INTERMEDIATE_FILE = "/content/drive/MyDrive/RA/health_care/dentist_LMS_keywords/output/final/clinic_yearly_nlp_traits.csv"
df_reviews_nlp = pd.read_csv(INTERMEDIATE_FILE)

ratio_to_binary = {
    'ad_ratio': 'is_ad_driven',
    'demo_ratio': 'target_demographic',
    'cosmetic_ratio': 'service_cosmetic',
    'ortho_ratio': 'service_ortho',
    'surgery_ratio': 'service_surgery',
    'price_issue_ratio': 'issue_price',
    'service_issue_ratio': 'issue_service',
    'compare_ratio': 'is_comparing'
}

print("Random 5 Review Samples per Category\n")

for ratio_col, binary_col in ratio_to_binary.items():
    print(f"[{ratio_col.upper()}] (Filter: {binary_col} == 1)")

    subset = df_reviews_nlp[df_reviews_nlp[binary_col] == 1]

    n_samples = min(5, len(subset))
    if n_samples > 0:
        # random_state=42
        samples = subset.sample(n=n_samples, random_state=42)['review_text'].tolist()
        for i, text in enumerate(samples, 1):
            clean_text = str(text).replace('\n', ' ').strip()
            display_text = f"{clean_text[:512]}..." if len(clean_text) > 512 else clean_text
            print(f"  {i}. {display_text}")
    else:
        print("  No reviews found for this category.")
    print("-" * 80)

In [ ]:
y_vars = ['dynamic_rating',
    'current_bad_review_pct',
    'price_issue_ratio',
    'service_issue_ratio',
    'compare_ratio',
    'ad_ratio',
    'demo_ratio',
    'cosmetic_ratio',
    'ortho_ratio',
    'surgery_ratio']

summary_stats_y = df_panel_valid[[var for var in y_vars if var in df_panel_valid.columns]].describe().T
summary_stats_y

In [ ]:
vars = ['log_density_25pct_total', 'log_density_25_to_50pct_total']
corr = df_panel_valid[vars].corr()
print(corr)

In [ ]:
import numpy as np
import statsmodels.api as sm
from linearmodels.panel import PanelOLS

panel_data = df_panel_valid.set_index(['clinic_id', 'year'])

dependent_vars = [
    'dynamic_rating',
    'current_bad_review_pct',
    'price_issue_ratio',
    'service_issue_ratio',
    'compare_ratio',
    'ad_ratio',
    'demo_ratio',
    'cosmetic_ratio',
    'ortho_ratio',
    'surgery_ratio'
]

for i, y_var in enumerate(dependent_vars, start=1):
    print(f"================================================================================")
    print(f"SET {i}: Dependent Variable -> {y_var}")
    print(f"================================================================================\n")

    print(f"Regression {i}.1: 0-25% + Total Entry")
    exog_vars_1 = ['log_lag_entry_shock_25pct', 'log_votes_dynamic', 'distance_to_hub']
    exog_1 = sm.add_constant(panel_data[exog_vars_1])
    mod_1 = PanelOLS(panel_data[y_var], exog_1, entity_effects=True, drop_absorbed=True)
    res_1 = mod_1.fit(cov_type='clustered', cluster_entity=True)
    print(res_1.summary)

    print(f"\nRegression {i}.2: 0-25% + Other Entry & Strong Entry")
    exog_vars_2 = ['log_other_entry_shock_25pct', 'log_strong_entry_shock_25pct', 'log_votes_dynamic', 'distance_to_hub']
    exog_2 = sm.add_constant(panel_data[exog_vars_2])
    mod_2 = PanelOLS(panel_data[y_var], exog_2, entity_effects=True, drop_absorbed=True)
    res_2 = mod_2.fit(cov_type='clustered', cluster_entity=True)
    print(res_2.summary)

    print(f"\nRegression {i}.3: 0-25% + Density")
    exog_vars_3 = ['log_density_25pct_total', 'log_votes_dynamic', 'distance_to_hub']
    exog_3 = sm.add_constant(panel_data[exog_vars_3])
    mod_3 = PanelOLS(panel_data[y_var], exog_3, entity_effects=True, drop_absorbed=True)
    res_3 = mod_3.fit(cov_type='clustered', cluster_entity=True)
    print(res_3.summary)

    print(f"\nRegression {i}.4: 0-25% & 25-50% + Total Entry")
    exog_vars_4 = ['log_lag_entry_shock_25pct', 'log_lag_entry_shock_25_to_50pct', 'log_votes_dynamic', 'distance_to_hub']
    exog_4 = sm.add_constant(panel_data[exog_vars_4])
    mod_4 = PanelOLS(panel_data[y_var], exog_4, entity_effects=True, drop_absorbed=True)
    res_4 = mod_4.fit(cov_type='clustered', cluster_entity=True)
    print(res_4.summary)

    print(f"\nRegression {i}.5: 0-25% & 25-50% + Other Entry & Strong Entry")
    exog_vars_5 = ['log_other_entry_shock_25pct', 'log_strong_entry_shock_25pct', 'log_other_entry_shock_25_to_50pct', 'log_strong_entry_shock_25_to_50pct', 'log_votes_dynamic', 'distance_to_hub']
    exog_5 = sm.add_constant(panel_data[exog_vars_5])
    mod_5 = PanelOLS(panel_data[y_var], exog_5, entity_effects=True, drop_absorbed=True)
    res_5 = mod_5.fit(cov_type='clustered', cluster_entity=True)
    print(res_5.summary)

    print(f"\nRegression {i}.6: 0-25% & 25-50% + Density")
    exog_vars_6 = ['log_density_25pct_total', 'log_density_25_to_50pct_total', 'log_votes_dynamic', 'distance_to_hub']
    exog_6 = sm.add_constant(panel_data[exog_vars_6])
    mod_6 = PanelOLS(panel_data[y_var], exog_6, entity_effects=True, drop_absorbed=True)
    res_6 = mod_6.fit(cov_type='clustered', cluster_entity=True)
    print(res_6.summary)
    print("\n\n")